<a href="https://colab.research.google.com/github/zach-yan/ORFE-Network-Project/blob/main/ORF_387_Project_Paper_Abstract_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Functions

In [ ]:
import requests
import pandas as pd
import time
import logging

# Set up logging to track errors without cluttering the output with print statements
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def fetch_author_papers(author_ids, api_key=None, batch_size=50, max_retries=5, save_path="author_papers_checkpoint.csv"):
    """Fetches titles and abstracts with exponential backoff and progress checkpointing."""
    headers = {"x-api-key": api_key} if api_key else {}
    base_url = "https://api.semanticscholar.org/graph/v1/author/batch"

    query_params = {
        "fields": "papers.title,papers.abstract,papers.year",
    }

    all_data = []
    failed_author_ids = []

    # Pre-clean IDs to prevent obvious 400 errors (e.g., removing None or empty strings)
    clean_ids = [str(aid) for aid in author_ids if pd.notna(aid) and str(aid).strip() != ""]

    for i in range(0, len(clean_ids), batch_size):
        batch = clean_ids[i:i+batch_size]
        success = False
        retries = 0
        backoff_time = 2  # Start with a 2-second pause for 429s

        while not success and retries < max_retries:
            try:
                response = requests.post(base_url, headers=headers, params=query_params, json={"ids": batch})

                if response.status_code == 200:
                    results = response.json()
                    for author in results:
                        if not author or 'papers' not in author:
                            continue

                        valid_papers = [p for p in author['papers'] if p.get('year') is not None]
                        recent_papers = sorted(valid_papers, key=lambda x: x['year'], reverse=True)[:5]

                        for paper in recent_papers:
                            if paper.get('title') and paper.get('abstract'):
                                all_data.append({
                                    "Author_Id": author['authorId'],
                                    "Paper_Id": paper['paperId'],
                                    "Title": paper['title'],
                                    "Abstract": paper['abstract'],
                                    "Text": f"{paper['title']}. {paper['abstract']}"
                                })

                    success = True
                    time.sleep(1)

                elif response.status_code == 429:
                    retries += 1
                    logging.warning(f"Rate limit (429) hit at batch {i}. Retrying in {backoff_time}s (Attempt {retries}/{max_retries}).")
                    time.sleep(backoff_time)
                    backoff_time *= 2

                # Catching 504s and other server-side timeouts
                elif response.status_code in [500, 502, 503, 504]:
                    retries += 1
                    logging.warning(f"Server error ({response.status_code}) at batch {i}. Retrying in {backoff_time}s (Attempt {retries}/{max_retries}).")
                    time.sleep(backoff_time)
                    backoff_time *= 2

                elif response.status_code == 400:
                    logging.error(f"400 Bad Request at batch {i}. Malformed ID likely present. Skipping batch.")
                    failed_author_ids.extend(batch)
                    break

                else:
                    logging.error(f"Unexpected {response.status_code} at batch {i}. Skipping.")
                    failed_author_ids.extend(batch)
                    break

            except requests.exceptions.RequestException as e:
                retries += 1
                logging.warning(f"Network error: {e}. Retrying in {backoff_time}s...")
                time.sleep(backoff_time)
                backoff_time *= 2

        if not success and retries == max_retries:
            logging.error(f"Max retries exceeded for batch {i}. Skipping.")
            failed_author_ids.extend(batch)

        # Checkpointing: Save progress to disk every 1000 authors processed
        if (i + batch_size) % 1000 == 0:
            pd.DataFrame(all_data).to_csv(save_path, index=False)
            logging.info(f"Checkpoint: {len(all_data)} total papers collected so far.")

    # Final conversion and save
    final_df = pd.DataFrame(all_data)
    final_df.to_csv(save_path, index=False)
    logging.info("Collection complete.")

    if failed_author_ids:
        logging.warning(f"Skipped {len(failed_author_ids)} IDs due to errors. See the returned list.")

    return final_df, failed_author_ids


Import the nodes from the previous notebook.

In [ ]:
nodes_df = pd.read_csv('orfe_nodes_fuzzy_cleaned.csv')
nodes_df.head()

,Id,Label,Layer,Affiliation
0,1414099319,&#x00C2;ndrei Camponogara,2.0,Unknown
1,1422426196,'. Gonz'alez,2.0,Unknown
2,2226784417,'Alex Hern'andez-Garc'ia,2.0,Unknown
3,2130393980,A. A.,2.0,Unknown
4,40804344,A. A. Abello,2.0,Unknown


Execution

API Key

In [ ]:
from google.colab import userdata
API_KEY = userdata.get('SemanticScholarAPI')

Execution

In [ ]:
author_list = nodes_df['Id'].tolist()

# Execution:
papers_df, bad_ids = fetch_author_papers(author_list, api_key=API_KEY)
papers_df.to_csv("author_papers_for_inference.csv", index=False)

ERROR:root:400 Bad Request at batch 0. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 1650. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 1950. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 3400. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 4900. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 4950. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 7600. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 7700. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 7750. Malformed ID likely present. Skipping batch.
ERROR:root:Max retries exceeded for batch 7800. Skipping.
ERROR:root:400 Bad Request at batch 8500. Malformed ID likely present. Skipping batch.
ERROR:root:400 Bad Request at batch 9650. Malformed ID likely present. Skip

Salvage more IDs if possible.

In [ ]:
def salvage_failed_ids(failed_ids, existing_df, api_key=None):
    """Processes failed IDs one by one to recover the collateral damage."""
    print(f"Attempting to salvage {len(failed_ids)} skipped IDs...")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Accept": "application/json"
    }
    if api_key and api_key != "YOUR_KEY" and str(api_key).strip() != "":
        headers["x-api-key"] = api_key

    base_url = "https://api.semanticscholar.org/graph/v1/author/batch"
    query_params = {"fields": "papers.title,papers.abstract,papers.year"}

    salvaged_data = []
    truly_bad_ids = []

    # Query one by one
    for aid in failed_ids:
        try:
            response = requests.post(base_url, headers=headers, params=query_params, json={"ids": [aid]})

            if response.status_code == 200:
                results = response.json()
                for author in results:
                    if not author or 'papers' not in author: continue

                    valid_papers = [p for p in author['papers'] if p.get('year') is not None]
                    recent_papers = sorted(valid_papers, key=lambda x: x['year'], reverse=True)[:5]

                    for paper in recent_papers:
                        if paper.get('title') and paper.get('abstract'):
                            salvaged_data.append({
                                "Author_Id": author['authorId'],
                                "Paper_Id": paper['paperId'],
                                "Title": paper['title'],
                                "Abstract": paper['abstract'],
                                "Text": f"{paper['title']}. {paper['abstract']}"
                            })
            else:
                truly_bad_ids.append(aid)

            time.sleep(0.5) # Gentle pacing

        except requests.exceptions.RequestException:
            truly_bad_ids.append(aid)

    # Combine the salvaged data with your main dataset
    salvaged_df = pd.DataFrame(salvaged_data)
    final_combined_df = pd.concat([existing_df, salvaged_df], ignore_index=True)

    print(f"Salvaged {len(salvaged_df)} valid papers.")
    print(f"Found {len(truly_bad_ids)} permanently invalid IDs. These can be safely ignored.")

    # Save the final complete dataset
    final_combined_df.to_csv("author_papers_final.csv", index=False)
    return final_combined_df

final_df = salvage_failed_ids(bad_ids, papers_df, api_key=API_KEY)

Attempting to salvage 1300 skipped IDs...
Salvaged 3152 valid papers.
Found 350 permanently invalid IDs. These can be safely ignored.


Download the model

In [ ]:
from transformers import pipeline

# 1. Load the pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 2. Save the underlying model and tokenizer to a new folder
classifier.model.save_pretrained("./bart_offline_model")
classifier.tokenizer.save_pretrained("./bart_offline_model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: bart_offline_model/ (stored 0%)
  adding: bart_offline_model/config.json (deflated 60%)
  adding: bart_offline_model/model.safetensors


zip error: Interrupted (aborting)


In [ ]:
# Create .tar file
!tar -cf bart_offline_model.tar bart_offline_model/

In [ ]:
# Download it for use on Della
from google.colab import files

files.download('bart_offline_model.tar')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>